In [1]:
import numpy as np
import pandas as pd
import yfinance as yf


In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [3]:
from services.market_service import load_live_chain, filter_chain_with_stats, parse_tickers

tickers = ['NVDA']
r = 0.05
q = 0.0
spread_limit = 0.05

# Raw chain — no filtering applied
raw_df = load_live_chain(tickers=parse_tickers(tickers))
print(f"Raw contracts: {len(raw_df)}")

# Apply all filters (same logic as the app)
filtered_df, filter_stats = filter_chain_with_stats(
    raw_df,
    spread_limit=spread_limit,
    r=r,
    q=q,
    tickers=parse_tickers(tickers),
    option_types=("call", "put"),
    min_volume=1,
    min_open_interest=0,
    max_maturity=2.0,
    # No max_contracts cap — see all contracts that pass quality filters
)
print(f"Filtered contracts: {len(filtered_df)}")
print("\nFilter breakdown:")
for reason, count in filter_stats.items():
    print(f"  {reason}: {count}")

filtered_df.head()

pulling...NVDA...


Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness'],
      dtype='object')
Raw contracts: 4097
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness', 'spot_over_strike', 'atm_distance',
       'contract_id'],
      dtype='object')
Filtered contracts: 1

Filter breakdown:
  Mid price ≤ 0.001: 3409
  Rel. spread ≥ 5%: 74
  Moneyness outside [0.8, 1.2]: 613


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,T,mid_price,rel_spread,moneyness,spot_over_strike,atm_distance,contract_id
0,NVDA270917P00245000,2026-03-19 15:34:27+00:00,245.0,78.18,61.85,62.25,0.0,0.0,10.0,95.0,...,208.270004,NVDA,american,1.391599,62.05,0.006446,1.176358,0.850082,0.162423,NVDA270917P00245000


## Data-Driven Heston Bounds — Dry Run (NVDA)

Estimate initial guess and tight search bounds for `(v0, kappa, theta, sigma, rho)`
directly from the market implied-vol surface, following:

| Param | Source |
|-------|--------|
| `v0`    | ATM IV² at shortest liquid maturity |
| `theta` | ATM IV² at longest liquid maturity  |
| `sigma` | vol-of-vol from smile curvature: `σ ≈ √(8c)` |
| `rho`   | correlation from ATM slope: `ρ ≈ 2b/σ` |
| `kappa` | mean reversion from term-structure slope |

In [5]:
import warnings
warnings.filterwarnings("ignore")
import sys
import os
sys.path.append(os.path.abspath(".."))
import pandas as pd

In [ ]:
from services.market_service import load_live_chain, parse_tickers
from analytics.schema import ensure_option_frame
from data.option_filters import apply_filters
from calibration.data_driven_bounds import compute_data_driven_bounds
from calibration.implied_vol import implied_volatility

In [14]:
# --- Load data ---------------------------------------------------------------
# Use sample Excel data so the dry run is reproducible regardless of market hours.
# The live filtered_df (from cell above) can be substituted if markets are open.

tickers = ['NVDA']
r = 0.05
q = 0.0
spread_limit = 0.05

# Raw chain — no filtering applied
df_raw = load_live_chain(tickers=parse_tickers(tickers))

df = ensure_option_frame(df_raw)

#Apply filteres
filtered_df , stats = apply_filters(
    df,
    spread_limit=0.10,
    r=0.05, q=0.0,
    moneyness_lo=0.7, moneyness_hi=1.3,
    min_volume=0, min_open_interest=0,
    keep_positive_time=True,        # live data
)

print(filtered_df.columns)
print(len(df_raw))
print(len(df))
print(len(filtered_df))

# Drop effectively-expired rows; keep only contracts with meaningful time value
filtered_df = filtered_df[filtered_df["T"] > 0.01].copy()

filtered_df.head()


pulling...NVDA...
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness'],
      dtype='object')
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness', 'spot_over_strike', 'atm_distance',
       'contract_id'],
      dtype='object')
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturit

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,T,mid_price,rel_spread,moneyness,spot_over_strike,atm_distance,contract_id
71,NVDA260504C00170000,2026-04-27 13:35:00+00:00,170.0,40.90,38.90,41.75,12.190002,42.459084,2.0,6.0,...,208.934998,NVDA,american,0.017947,40.325,0.070676,0.813650,1.229029,0.206225,NVDA260504C00170000
72,NVDA260504C00172500,2026-04-23 18:50:52+00:00,172.5,26.88,37.10,38.80,0.000000,0.000000,NaN,11.0,...,208.934998,NVDA,american,0.017947,37.950,0.044796,0.825616,1.211217,0.191626,NVDA260504C00172500
73,NVDA260504C00175000,2026-04-27 14:08:00+00:00,175.0,35.71,34.45,36.00,0.809998,2.320910,1.0,9.0,...,208.934998,NVDA,american,0.017947,35.225,0.044003,0.837581,1.193914,0.177237,NVDA260504C00175000
74,NVDA260504C00177500,2026-04-24 19:32:58+00:00,177.5,31.25,32.15,33.80,0.000000,0.000000,4.0,8.0,...,208.934998,NVDA,american,0.017947,32.975,0.050038,0.849547,1.177099,0.163053,NVDA260504C00177500
75,NVDA260504C00180000,2026-04-27 14:06:26+00:00,180.0,30.65,29.80,30.70,1.750000,6.055363,7.0,32.0,...,208.934998,NVDA,american,0.017947,30.250,0.029752,0.861512,1.160750,0.149066,NVDA260504C00180000


In [31]:
result = compute_data_driven_bounds(
    filtered_df,
    r=0.05,
    q=0.0,
    atm_threshold=0.10,   # |log(K/S)| < 10% for ATM IV estimation
    fit_threshold=0.15,   # |log(K/S)| < 15% for skew/curvature regression
    min_contracts=5,
)

In [32]:

#sample_path = "../nvda_vol.xlsx"
#df_raw = pd.read_excel(sample_path)
df = ensure_option_frame(df_raw)

sample_df, stats = apply_filters(
    df,
    spread_limit=0.10,
    r=0.05, q=0.0,
    moneyness_lo=0.7, moneyness_hi=1.3,
    min_volume=0, min_open_interest=0,
    keep_positive_time=False,        # stored T, not live
)
# Drop effectively-expired rows; keep only contracts with meaningful time value
sample_df = sample_df[sample_df["T"] > 0.01].copy()

print(f"Sample: {len(sample_df)} contracts  |  {sample_df['T'].nunique()} unique maturities  |  "
      f"T ∈ [{sample_df['T'].min():.3f}, {sample_df['T'].max():.3f}] yr")

# --- Compute data-driven bounds ----------------------------------------------
result = compute_data_driven_bounds(
    sample_df,
    r=0.05,
    q=0.0,
    atm_threshold=0.10,   # |log(K/S)| < 10% for ATM IV estimation
    fit_threshold=0.15,   # |log(K/S)| < 15% for skew/curvature regression
    min_contracts=5,
)

ig   = result["initial_guess"]
bnds = result["bounds"]
diag = result["diagnostics"]

# --- Print results -----------------------------------------------------------
print("\n" + "=" * 65)
print("  NVDA — Data-Driven Heston Initial Guess & Bounds")
print("=" * 65)

if "warning" in diag:
    print(f"  WARNING: {diag['warning']}")
    print("  (Showing static fallback defaults)")
else:
    labels = [
        "v0     initial variance",
        "kappa  mean reversion  ",
        "theta  long-run variance",
        "sigma  vol-of-vol      ",
        "rho    correlation     ",
    ]
    for label, guess, (lo, hi) in zip(labels, ig, bnds):
        print(f"  {label}  guess = {guess:+.4f}   bound = [{lo:+.4f}, {hi:+.4f}]")

    print()
    print("  ── Surface diagnostics ──────────────────────────────────")
    print(f"  Short maturity  T_short      = {diag['T_short']:.4f} yr  "
          f"({diag['T_short']*365:.1f} days)")
    print(f"  Long  maturity  T_long       = {diag['T_long']:.4f} yr  "
          f"({diag['T_long']*365:.1f} days)")
    print(f"  ATM IV  (T_short)            = {diag['sigma_atm_short']:.4f}  "
          f"({diag['sigma_atm_short']*100:.2f} %)")
    print(f"  ATM IV  (T_long )            = {diag['sigma_atm_long']:.4f}  "
          f"({diag['sigma_atm_long']*100:.2f} %)")
    print(f"  Term-structure slope         = {diag['slope_T']:+.4f} vol / yr")
    print(f"  Smile slope  b  (skew)       = {diag['smile_slope_b']:+.4f}")
    print(f"  Smile curvature c            = {diag['smile_curvature_c']:+.4f}")
    print(f"  Liquid maturities  ({diag['n_liquid_maturities']:2d}):      "
          f"{[round(t, 3) for t in diag['liquid_maturities']]}")

print("=" * 65)

Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness', 'spot_over_strike', 'atm_distance',
       'contract_id'],
      dtype='object')
Sample: 1232 contracts  |  22 unique maturities  |  T ∈ [0.018, 2.637] yr

  NVDA — Data-Driven Heston Initial Guess & Bounds
  v0     initial variance  guess = +0.1820   bound = [+0.1274, +0.2366]
  kappa  mean reversion    guess = +0.3818   bound = [+0.5000, +1.1454]
  theta  long-run variance  guess = +0.2202   bound = [+0.1101, +0.4403]
  sigma  vol-of-vol        guess = +2.0000   bound = [+1.0000, +2.0000]
  rho    correlation       guess = -0.8676   bound = [-0.9990, -0.6676]

  ── Surface diagnostics ──────────────────────────────────
  Short maturity  T_short      = 0.0179 yr  (6.6